# Week 1: Introduction to Genetic Risk Scores

## This week at a glance

**Big idea:** This week I learn what a polygenic risk score (PRS) is: a number that adds up the small effects of many genetic variants to estimate a person's genetic risk for a trait or disease. The hope is that a PRS could flag high-risk people early, before symptoms, which could improve preventive care.

**My readings**
| Paper | What I read | Why it matters |
|---|---|---|
| Purcell et al. 2009, *Nature* | Up to the paragraph about Figure 2 | First paper to build a polygenic risk score |
| Khera et al. 2018, *Nature Genetics* | Up to the paragraph about Table 2 | First paper to argue PRS could be useful in the clinic |
| Martin et al. 2019, *Nature Genetics* | All main text | Shows PRS work much worse for non-European ancestries |

**My tasks**
1. What are the file extensions of the genotype data vs. the expression data?
2. What are the different fields of the genotype data?
3. What software do geneticists commonly use for these files?

**Weekly questions I answer on Gradescope**
1. In what ways can polygenic risk scores be used for preventive care?
2. What are your two favorite reasons PRS are not ready to be used in the clinic yet?

**My data**
- Genotypes: 1000 Genomes European samples (`../data/raw/LDREF/`)
- Gene expression: Geuvadis blood cell RNA-seq (`../data/raw/GD462.GeneQuantRPKM.50FN.samplename.resk10.txt`)

## Biology background and vocabulary

### The cookbook analogy

| Analogy | Real term | What it actually is |
|---|---|---|
| The whole cookbook | **DNA** | The instructions inside every cell, written with 4 letters: A, C, G, T (about 3 billion letters long) |
| The entire cookbook, cover to cover | **Genome** | All of one person's DNA: every chromosome, every gene, and the DNA between genes. "Genome-wide" means across all of it |
| Chapters of the book | **Chromosomes** | DNA is split into 23 chromosomes. Every cell has two copies of each (46 total) |
| One recipe | **Gene** | A stretch of DNA that makes one specific thing, usually a protein. Humans have about 20,000 protein-making genes |
| Two copies of the book | **Two copies of DNA** | One copy inherited from mom, one from dad. So every gene, and every spot in the DNA, comes in two copies |
| A typo some people have | **SNP** ("snip") | One spot in the DNA where people can have different letters |
| The two possible letters at that spot | **Alleles** | Example: a SNP where people have either C or T has two alleles, C and T |
| A person's two letters at that spot | **Genotype** | Example: CT (one C, one T). The possibilities are CC, CT, or TT |
| Volume knob next to a recipe | **Regulatory region** | DNA near a gene that controls how much the gene gets used |
| Photocopy of a recipe | **RNA** | A temporary copy the cell makes so it can use a gene |
| The finished dish | **Protein** | The working part the cell builds from the RNA |
| How many photocopies get made | **Gene expression** | How much a cell is using a gene |

### Key facts

- **Every cell has every gene, but only uses some of them.** A gut cell has the eye color genes but doesn't express them.
- **People's DNA is about 99.9% identical.** The 0.1% that differs, mostly SNPs, is part of what makes each person unique, including differences in disease risk.
- **How DNA is passed down:** each parent passes on one copy, made by stitching together pieces of their own two copies. The switch points are called **recombination** ("cuts"). SNPs that sit close together rarely get split, so they tend to be inherited together.

### Seeing SNPs in the DNA

**Both sides of the ladder.** DNA is a double helix, like a twisted ladder. Across the ladder, letters (called **bases**) always pair A with T and C with G, so the bottom side is a mirror of the top. Data files read only the top side (the **reference strand**).

```
                     position:  1  2  3  4
Mom's chr 1   top side:         A  T  G  C
              bottom side:      T  A  C  G

Dad's chr 1   top side:         A  T  A  C
              bottom side:      T  A  T  G
                                      ^
                               SNP at position 3
```

Reading the top side at position 3: mom's copy has G and dad's has A, so the genotype is GA. Reading the bottom side would give CT, which is the same answer from the other side (C mirrors G, T mirrors A).

**Each SNP has exactly two options, but different SNPs can have different pairs.** Top side only:

```
                     position:  1  2  3  4  5  6  7  8
Person 1   Mom's copy:          A  T  G  C  T  T  A  G
           Dad's copy:          A  T  G  C  T  T  A  G

Person 2   Mom's copy:          A  T  G  C  T  T  C  G
           Dad's copy:          A  T  A  C  T  T  A  G

Person 3   Mom's copy:          A  T  A  C  T  T  C  G
           Dad's copy:          A  T  A  C  T  T  C  G
                                      ^           ^
                                    SNP 1       SNP 2
```

| | SNP 1 (position 3) | SNP 2 (position 7) |
|---|---|---|
| Person 1 | GG | AA |
| Person 2 | GA | CA |
| Person 3 | AA | CC |

At SNP 1, the only letters anyone has are G and A. At SNP 2, the only letters are A and C. Every other position is the same for everyone, so those aren't SNPs. Each SNP's pair comes from a single typo that happened at that spot long ago.

### How genotypes become numbers

Data files don't store letters. For each SNP, they pick one allele to count (the **counted allele**) and store how many copies each person has:

| Genotype (counting C) | Number stored |
|---|---|
| TT | 0 |
| CT | 1 |
| CC | 2 |

### A real example: drinking milk (the LCT gene)

The gene **LCT** makes lactase, the protein that breaks down milk sugar. Next to LCT is a regulatory region with a SNP where people have C or T:

- **CC:** the knob turns down after childhood, so little lactase is made, and the person usually can't digest milk well (lactose intolerant)
- **CT or TT:** the knob stays up, so lactase keeps being made, and the person can usually drink milk as an adult

This is the full chain the whole capstone is built on:

**SNP letter → regulatory region turned up or down → gene expression → protein → trait**

### Week 1 terms

| Term | Meaning |
|---|---|
| **Trait** | Something measurable about a person, like height or whether they have a disease |
| **Polygenic** | Affected by many genes ("poly" = many). Height is affected by thousands of SNPs, each with a tiny effect |
| **Pleiotropy** | The opposite direction: one gene or SNP affecting more than one trait |
| **GWAS** (genome-wide association study) | A study that tests every SNP in the genome to see which ones are linked to a trait |
| **Effect size** | How much each extra copy of an allele pushes the trait up or down, estimated by a GWAS |
| **Polygenic risk score (PRS)** | A person's score from adding up, across many SNPs, copies of the allele x that allele's effect size |
| **Ancestry** | Where a person's recent family history comes from (for example European, African, East Asian). It matters because SNP patterns differ between populations |

## Polygenic risk scores (PRS)

### The idea

For polygenic traits like height or heart disease, thousands of SNPs each give a tiny push up or down. A PRS adds up all of a person's pushes into one number.

**Formula:** for each SNP, multiply the person's copies of the counted allele (0, 1, or 2) by that allele's effect size, then add everything up.

PRS = (copies at SNP 1 x effect size 1) + (copies at SNP 2 x effect size 2) + ...

The effect sizes come from a **GWAS** done in a large group of other people.

### Worked example

| SNP | Effect size (from a GWAS) | Adam's copies | Adam's push | Sara's copies | Sara's push |
|---|---|---|---|---|---|
| SNP A | +2 | 2 | +4 | 0 | 0 |
| SNP B | -1 | 1 | -1 | 2 | -2 |
| SNP C | +0.5 | 0 | 0 | 2 | +1 |
| **PRS** | | | **+3** | | **-1** |

Adam has more genetic pushes toward the trait than Sara. A real PRS uses thousands of SNPs.

### A score only means something compared to other people

A score of +3 on its own doesn't say much. So I compare it to many other people's scores and see where that person lands. Across a population, PRS values usually form a **bell curve**, because each person gets a random mix of ups and downs that mostly cancel out. A person is described by their **percentile**: at the 90th percentile, their score is higher than 90% of people.

### Why PRS could help preventive care

- **DNA doesn't change**, so a PRS can be calculated at any age, even at birth, long before symptoms.
- People at the high end could start **screening earlier** (like cholesterol checks or mammograms) or make **lifestyle changes** sooner.
- Khera et al. (2018) found that about **8%** of people carry roughly **3 times** the normal risk of coronary artery disease from polygenic risk alone, a level of risk similar to rare single-gene mutations doctors already screen for. Many more people fall into this group than carry those rare mutations.

### Why PRS are not ready for the clinic yet

- **They work much worse outside European ancestry.** Most GWAS participants have been European, so the effect sizes don't transfer well. Martin et al. (2019) found accuracy was about **2.5 times lower** in East Asian and about **4.9 times lower** in African ancestry populations. Using PRS now could widen health gaps.
- **A PRS is a probability, not a diagnosis.** A high score doesn't mean someone will get the disease, and most people fall in the middle, where the score doesn't change what a doctor would do.
- **Genes aren't everything.** Environment and lifestyle also matter, and a PRS only captures part of the genetic effect.
- **Little proof yet that it improves outcomes.** There isn't much evidence that acting on a PRS actually helps patients, and results can be misunderstood (for example, someone with a "low risk" score skipping screening).

## Task: Getting to know the data

### Step 1: What files are in the genotype folder?

In [ ]:
import os

genotype_folder = "../data/raw/LDREF"
genotype_file_names = sorted(os.listdir(genotype_folder))

print("Number of files:", len(genotype_file_names))
genotype_file_names[:9]

**What I see:** I found 66 files: 22 chromosomes x 3 file types (chromosome 23, the sex chromosome, is not included). Each name follows the pattern `1000G.EUR.<chromosome>.<type>`: `1000G` is the 1000 Genomes Project, `EUR` means only people of European ancestry, and the number is the chromosome.

| File type | Survey version | What's inside |
|---|---|---|
| `.fam` | List of people who took the survey | One row per person |
| `.bim` | List of questions | One row per SNP: its ID, location, and two possible alleles |
| `.bed` | Everyone's answers | The 0, 1, 2 counts for every person at every SNP, stored in a compressed format |

### Step 2: The `.fam` file (the list of people)

In [ ]:
import pandas as pd

fam_column_names = ["family_id", "person_id", "dad_id", "mom_id", "sex", "trait_value"]

people_table = pd.read_csv(
    "../data/raw/LDREF/1000G.EUR.1.fam",
    sep=r"\s+",
    header=None,
    names=fam_column_names,
)

print("Number of people:", len(people_table))
people_table.head()

**What I see:** I see 489 people, one row each. `person_id` is a code like HG00096 instead of a real name, and `family_id` matches it because each person is their own family here. `dad_id` and `mom_id` are 0 because no parents are in the data (only unrelated people were included). `sex` is 0 (not recorded), and `trait_value` is -9, the file's code for missing. That's fine for this project, since the trait I study (gene expression) comes from a separate file.

### Step 3: The `.bim` file (the list of SNPs)

In [ ]:
bim_column_names = ["chromosome", "snp_id", "genetic_distance", "position", "counted_allele", "other_allele"]

snp_table = pd.read_csv(
    "../data/raw/LDREF/1000G.EUR.1.bim",
    sep=r"\s+",
    header=None,
    names=bim_column_names,
)

print("Number of SNPs on chromosome 1:", len(snp_table))
snp_table.head()

**What I see:** I see that chromosome 1 has 98,642 SNPs, one row each, and no people. This file is the list of SNP spots that everyone was measured at. `snp_id` is each SNP's unique ID (like rs3094315), and `position` is its letter number counting from the start of the chromosome. `counted_allele` is the allele that gets counted in the `.bed` file, and `other_allele` is the second option at that SNP. For example, at rs3094315 the alleles are G and A, so someone with genotype GA is stored as 1. Each SNP has its own pair of alleles (G/A, C/T, A/C, and so on). `genetic_distance` also describes location, but in a unit based on how often recombination happens, so I use `position` instead.

### Step 4: The gene expression file

This is the second dataset, from the Geuvadis project, which measured gene expression in blood cells grown in a lab (lymphoblastoid cell lines, or LCLs) from many of the same 1000 Genomes people.

In [ ]:
expression_path = "../data/raw/GD462.GeneQuantRPKM.50FN.samplename.resk10.txt"

expression_table = pd.read_csv(expression_path, sep="\t")

print("Number of rows:", expression_table.shape[0])
print("Number of columns:", expression_table.shape[1])

# First 5 rows and first 8 columns, since the table is very wide
expression_table.iloc[:5, :8]

**What I see:** I see 23,722 rows and 466 columns. Each row is one gene (about 20,000 of them make proteins, and the rest do other jobs without making a protein). The first 4 columns describe the gene: `TargetID` is its ID code, `Gene_Symbol` repeats the ID here, `Chr` is the chromosome it's on, and `Coord` is its position on that chromosome. The other 462 columns are people, using the same ID codes as the genotype files (like HG00096). Each number is how much that person's cells use (express) that gene. The values have been adjusted to remove technical noise, so they're relative amounts and some are negative. This file is about the genes themselves. The regulatory regions (knobs) that control them aren't listed here.

## Readings guide

I read them in this order. For each one, I skim the abstract first, then read the assigned part with the guiding questions in mind, and write my answers under **My notes**. I don't need to understand every term, so I focus on the questions.

---

### 1. Purcell et al. 2009, *Nature*: the first polygenic risk score

**What it's about:** Researchers studying schizophrenia took effect sizes from one group of people (the "discovery" sample) and used them to score a second, separate group (the "target" sample). Even though no single SNP was strong on its own, adding up thousands of tiny effects predicted who had the disease better than chance.

**My notes:**
1. Scoring the same people the effect sizes came from would overfit, since the effect sizes partly reflect luck in that group. Testing on a separate group (they used men to build the score and women to test it) shows whether the score really generalizes.
2. Looser cutoffs helped. The score explained more variance as more SNPs were included, up to about 3% at p < 0.5 (37,655 SNPs), because many real effects are too small to reach low p-values.
3. Yes, it also predicted bipolar disorder, which is pleiotropy. It did not predict non-psychiatric diseases like heart disease or diabetes. It also worked much worse in African-Americans (0.4% vs. about 3% in Europeans).

---

### 2. Khera et al. 2018, *Nature Genetics*: PRS in the clinic

**What it's about:** Using a very large dataset (UK Biobank), the authors built polygenic scores for five common diseases, including coronary artery disease, and asked how many people have high genetic risk.

**My notes:**
1. Coronary artery disease (CAD), atrial fibrillation (AF), type 2 diabetes, inflammatory bowel disease (IBD), and breast cancer.
2. 8% of people had 3 times or more the normal risk of CAD based on their score alone.
3. The rare FH mutation gives similar (up to 3x) CAD risk but is carried by only 0.4% of people, so the polygenic score finds about 20 times more people at that level of risk. Since doctors already screen for FH, this suggests polygenic scores could identify many more high-risk people who would benefit from early prevention. Limitation: everyone in the study was of British ancestry.

---

### 3. Martin et al. 2019, *Nature Genetics*: PRS and health gaps

**What it's about:** The authors show that most people in GWAS are of European ancestry, so polygenic scores built from those studies work much worse for other populations.

**My notes:**
1. About 79% of GWAS participants are of European ancestry, even though Europeans are only about 16% of the world's population.
2. Compared to Europeans, scores were about 2.5 times less accurate in East Asians and 4.9 times less accurate in Africans (also 1.6 times for Hispanic/Latino and 1.7 times for South Asian groups).
3. Diversify GWAS (the most important step), protect against genetic discrimination, share GWAS results openly across ancestries, build diverse resources like reference panels and gene expression data, fund research in lower-income countries, and standardize methods.


## Week 1 answers

### Task questions

**1. What are the file extensions of the genotype data vs. the expression data?**
The genotype data uses PLINK files: `.bed`, `.bim`, and `.fam`, one set per chromosome. The expression data is a single tab-separated `.txt` file.

**2. What are the different fields of the genotype data?**
- `.fam` (people): family ID, person ID, dad's ID, mom's ID, sex, and trait value
- `.bim` (SNPs): chromosome, SNP ID, genetic distance, position, counted allele, and other allele
- `.bed`: the 0, 1, 2 counts for every person at every SNP, stored in a compressed format

**3. What software is commonly used to work with these file types?**
PLINK (and its newer version, plink2). In code, packages like plink2R (R) and bed-reader (Python) can read these files.

### Gradescope questions

**1. In what ways can polygenic risk scores be used for preventive care?**
A polygenic risk score adds up the small effects of many genetic variants to estimate a person's genetic risk for a disease. Since DNA doesn't change, the score can be calculated early in life, long before any symptoms appear. Doctors could use it to find people at high risk and act early, like starting cholesterol checks or statins sooner for heart disease, or starting cancer screening at a younger age. For example, Khera et al. (2018) found that about 8% of people have about three times the normal risk of heart disease from their polygenic risk alone. Finding these people early could help prevent disease instead of only treating it later.

**2. What are your two favorite reasons PRS are not ready to be used in the clinic yet?**
My first reason is that PRS work much worse for people who aren't of European ancestry. Most GWAS have been done mostly in European participants, so the effect sizes don't transfer well to other groups. Martin et al. (2019) found prediction accuracy was about 2.5 times lower in East Asian and about 4.9 times lower in African ancestry populations. If PRS were used in the clinic now, people of European ancestry would get more accurate predictions and more benefit, so existing health gaps would grow.

My second reason is that a PRS gives a probability, not a diagnosis. Most people land in the middle of the distribution, where the score doesn't change what a doctor would do, and even a high score doesn't mean someone will get the disease. There also isn't much evidence yet that acting on a PRS improves health outcomes, and patients could misread results, like someone with a "low risk" score skipping screening.